# 07 — Uncertainty sampling

**Code under test:** `fleximorpv2/uncertainty_analysis.py`

**Purpose:** Compare Monte Carlo vs Latin Hypercube and confirm scenarios move LCOE.

Run cells **top to bottom**. Each markdown cell explains the **next code cell** — what it does, what to inspect in the output, and what counts as a pass.

Track overall audit progress in Obsidian (`FlexiMORP Calculation Audit.md`). These notebooks are the lab workbook, not the checklist.

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("error", category=RuntimeWarning)


def _repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "fleximorpv2").is_dir():
            return candidate
    raise RuntimeError(
        "Could not find fleximorp-project root. "
        "Open Jupyter from the repo or a notebooks/ subdirectory."
    )


_repo = _repo_root()
_audit = _repo / "notebooks" / "audit"
for path in (_repo, _audit):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from _audit_helpers import (
    REPO_ROOT,
    OUTPUT_DIR,
    SITE_OUTPUT_DIR,
    reload_fleximorp,
    assert_close,
    assert_energy_balance,
    assert_cf_bounds,
    assert_positive,
)

reload_fleximorp()
np.random.seed(42)
print(f"Repo root: {REPO_ROOT}")
print(f"Audit outputs: {OUTPUT_DIR}")
print(f"Site outputs: {SITE_OUTPUT_DIR}")

## Step 1 — Sampling method comparison

**Run the next cell** (small `n_runs=50` for speed).

**Pass if:**
- Both methods complete without error
- LCOE distribution has non-zero spread when uncertainty params are enabled
- LHS mean LCOE is in the same ballpark as MC (not identical, but same order of magnitude)

In [ ]:
from fleximorpv2.config import load_config
from fleximorpv2.baseline_optimization import BaselineOptimization
from fleximorpv2.uncertainty_analysis import UncertaintyAnalysis

config = load_config("alaska")
config.uncertainty["monte_carlo_runs"] = 50
baseline = BaselineOptimization(config).optimize("production", 200_000, method="scipy")
analyzer = UncertaintyAnalysis(config)

comparison = analyzer.compare_sampling_methods(
    baseline_design=baseline.optimal_design,
    n_runs=50,
)
print(comparison["convergence_analysis"])